## **SXO Keyword Opportunity** <br>
Eric Weldon

**Question:** A B2B company with a small content team can't write for every keyword. Which ones are already ours, which are open, which are worth fighting a competitor for, and which aren't worth the effort?

This notebook shows the core method of a search research tool I built to plan website copy and content for a B2B IoT company. The full tool crawls live sites, pulls Google Autocomplete and Brave Search rankings, and has an LLM judge fit and intent. Here the same scoring runs on a small built-in sample for a fictional company, **Example Co**, which sells inventory forecasting software, so it runs top to bottom with no API keys or internet.

1. **Distinctive phrases:** which phrases each site emphasizes, from field-weighted TF-IDF
2. **Demand, fit and ranking:** how much each keyword is searched, how well it fits what we sell, and who ranks
3. **Labels and opportunity:** ours, open, fight for, not winnable or not worth it, and a score to rank them
4. **Draft check:** an automated check that a draft page targets its keyword the way search engines and AI answers reward

In [1]:
import math
import re
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_colwidth", 80)

## 1. Which phrases does each site emphasize?

**Data:** a few pages from Example Co and two fictional competitors. The full tool crawls up to 40 pages per site and reads each page's title, URL, headings, link text and body.

**Approach:** each site's pages are treated as one document, and every 1 to 3 word phrase is counted. Where a phrase appears matters: a phrase in a title or URL counts 5 times, an H1 3 times, a subheading or link text twice, and body text once. TF-IDF then scores a phrase high for a site when that site uses it prominently **and the other sites don't**. That's what makes it the site's positioning, not just common industry language.

In [2]:
SITES = {
    "Example Co": [
        {"title": "Inventory Forecasting Software for Multi-Location Retail", "url": "/inventory-forecasting",
         "h1": "Forecast demand for every store and every item",
         "subheadings": ["Reduce stockouts without adding safety stock", "Automated reorder suggestions"],
         "body": "Example Co forecasts demand per store and per item, so each location orders what it will sell. "
                 "Retailers reduce stockouts and lower inventory carrying costs."},
        {"title": "How to Reduce Stockouts Across Multiple Stores", "url": "/guides/reduce-stockouts",
         "h1": "How to reduce stockouts across multiple stores",
         "subheadings": ["What causes stockouts?", "How much safety stock do you need?"],
         "body": "Stockouts happen when a store forecast is too low. Store-level forecasting fixes the cause."},
    ],
    "Competitor A": [
        {"title": "Warehouse Management System | Competitor A", "url": "/warehouse-management",
         "h1": "One warehouse management system for every warehouse",
         "subheadings": ["Barcode scanning", "Inventory management for multiple warehouses"],
         "body": "Track inventory across warehouses with barcode scanning and cycle counts."},
    ],
    "Competitor B": [
        {"title": "Demand Planning Software for Enterprise Supply Chains", "url": "/demand-planning",
         "h1": "Demand planning for global supply chains",
         "subheadings": ["Inventory optimization", "Safety stock calculation"],
         "body": "Plan demand across regions and optimize inventory for the whole supply chain."},
    ],
}
FIELD_WEIGHTS = {"title": 5, "url": 5, "h1": 3, "subheadings": 2, "body": 1}
STOP = set("""a an and are as at be by can do does for from how i if in is it my of on or our so that the their
them this to we what when where which who why will with you your every each one per across whole""".split())


def stem(word):
    """Merge plural and tense variants: 'stockouts' and 'stockout' count as one phrase."""
    for suffix in ("ing", "ers", "ies", "es", "ed", "er", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 4:
            return word[: -len(suffix)]
    return word


def phrases(text, n_max=3):
    """1 to 3 word phrases that don't start or end with a stopword or cross punctuation."""
    out = []
    for chunk in re.split(r"[.!?;:,()|/\-]+", text.lower()):
        words = re.findall(r"[a-z][a-z0-9']*", chunk)
        for n in range(1, n_max + 1):
            for i in range(len(words) - n + 1):
                gram = words[i:i + n]
                if gram[0] in STOP or gram[-1] in STOP or (n == 1 and len(gram[0]) < 4):
                    continue
                out.append(" ".join(gram))
    return out


def weighted_counts(pages):
    counts, surface = Counter(), defaultdict(Counter)
    for page in pages:
        for field, weight in FIELD_WEIGHTS.items():
            texts = page[field] if isinstance(page[field], list) else [page[field].replace("/", " ")]
            for text in texts:
                for p in phrases(text.replace("-", " ") if field == "url" else text):
                    key = " ".join(stem(w) for w in p.split())
                    counts[key] += weight
                    surface[key][p] += 1
    return counts, {k: v.most_common(1)[0][0] for k, v in surface.items()}


counts, surfaces = {}, {}
for site, pages in SITES.items():
    counts[site], surfaces[site] = weighted_counts(pages)
total = Counter()
for c in counts.values():
    total.update(c)
avg_size = sum(sum(c.values()) for c in counts.values()) / len(counts)

rows = []
for site, c in counts.items():
    size = sum(c.values())
    for key, w in c.items():
        if w >= 4 and len(key.split()) >= 2:
            score = (w / size) * math.log(1 + avg_size / total[key]) * 1000
            rows.append({"site": site, "phrase": surfaces[site][key], "weight": w,
                         "share of all uses": round(w / total[key], 2), "score": round(score, 1)})

distinctive = (pd.DataFrame(rows).sort_values(["site", "score"], ascending=[True, False])
               .groupby("site").head(5).reset_index(drop=True))
distinctive

,site,phrase,weight,share of all uses,score
0,Competitor A,warehouse management,13,1.0,333.9
1,Competitor A,management system,8,1.0,240.8
2,Competitor A,warehouse management system,8,1.0,240.8
3,Competitor B,demand planning,13,1.0,248.1
4,Competitor B,supply chains,9,1.0,194.0
5,Competitor B,planning software,5,1.0,128.0
6,Competitor B,enterprise supply,5,1.0,128.0
7,Competitor B,demand planning software,5,1.0,128.0
8,Example Co,reduce stockouts,16,1.0,141.2
9,Example Co,inventory forecasting,10,1.0,103.8


**Reading it:** each site's top phrases describe its positioning. Example Co's are about reducing stockouts and inventory forecasting, Competitor A's about warehouse management, and Competitor B's about enterprise demand planning. A share of 1.0 means only that site uses the phrase, so it's distinctive to them, not shared industry language. Phrases like these become candidate keywords, next to the seed keywords and the differentiators the company gives the tool.

## 2. Demand, fit and who ranks

**Data:** a research snapshot for each candidate keyword. In the full tool these come from live sources:

- **Demand:** Google Autocomplete. Google only completes queries people actually type, and popular queries grow many longer variants, so the number of variants is a relative demand signal. It isn't search volume.
- **Ranking:** the top results from the Brave Search API, and where each tracked site ranks.
- **Intent:** who the top results are written for (business buyers, consumers, researchers or job seekers). An LLM judges this from the result titles; here it's given.

**Fit** is how well a keyword matches what the company sells: the share of the keyword's words found in its closest differentiator, discounted when the keyword names none of the company's product terms. In the full tool, an LLM judges fit by meaning and explains its reason.

In [3]:
DIFFERENTIATORS = [
    "inventory forecasting software for multi-location retailers",
    "forecast demand per store and per item",
    "reduce stockouts without adding safety stock",
    "lower inventory carrying costs",
    "automated reorder suggestions",
]
PRODUCT_TERMS = ["inventory", "forecast", "stockout", "reorder", "safety stock", "carrying cost"]

# keyword, autocomplete variants, exact completion, intent, our rank, Competitor A rank, Competitor B rank
SNAPSHOT = [
    ("inventory forecasting software", 8, True, "business", None, 7, 5),
    ("store level demand forecasting", 2, True, "business", 4, None, None),
    ("reduce stockouts", 6, True, "business", None, None, None),
    ("how to reduce stockouts in retail", 3, True, "business", None, None, 12),
    ("inventory carrying costs", 9, True, "research", None, None, 9),
    ("safety stock calculation", 10, True, "research", None, None, 2),
    ("automated reorder point", 4, True, "business", None, 6, None),
    ("inventory management for multiple warehouses", 5, True, "business", None, 1, None),
    ("warehouse management system", 10, True, "business", None, 2, None),
    ("inventory clerk jobs", 10, True, "jobs", None, None, None),
    ("home pantry inventory app", 7, True, "consumer", None, None, None),
]
research = pd.DataFrame(SNAPSHOT, columns=["keyword", "variants", "exact", "intent", "our_rank", "a_rank", "b_rank"])


def words(text):
    return {stem(w) for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 2}


def demand(variants, exact):
    """0.45 for a query Google completes, plus 0.055 per longer variant (up to 10), capped at 1."""
    return min(1.0, 0.45 + 0.055 * min(variants, 10)) if (exact or variants) else 0.1


def fit(keyword):
    kw = words(keyword)
    overlap = max(len(kw & words(d)) / len(kw) for d in DIFFERENTIATORS)
    names_product = any(words(t) <= kw for t in PRODUCT_TERMS)
    return overlap * (1.0 if names_product else 0.35)


research["demand"] = [round(demand(v, e), 2) for v, e in zip(research.variants, research.exact)]
research["fit"] = [round(fit(k), 2) for k in research.keyword]
research[["keyword", "demand", "fit", "intent", "our_rank", "a_rank", "b_rank"]]

,keyword,demand,fit,intent,our_rank,a_rank,b_rank
0,inventory forecasting software,0.89,1.00,business,NaN,7.0,5.0
1,store level demand forecasting,0.56,0.75,business,4.0,NaN,NaN
2,reduce stockouts,0.78,1.00,business,NaN,NaN,NaN
3,how to reduce stockouts in retail,0.61,0.67,business,NaN,NaN,12.0
4,inventory carrying costs,0.95,1.00,research,NaN,NaN,9.0
5,safety stock calculation,1.00,0.67,research,NaN,NaN,2.0
6,automated reorder point,0.67,0.67,business,NaN,6.0,NaN
7,inventory management for multiple warehouses,0.73,0.25,business,NaN,1.0,NaN
8,warehouse management system,1.00,0.00,business,NaN,2.0,NaN
9,inventory clerk jobs,1.00,0.33,jobs,NaN,NaN,NaN


## 3. Labels and opportunity

**Labels** decide what to do with each keyword:

| Label | Rule |
|---|---|
| **not worth it** | Fit under 0.4, consumer or job-seeker intent, or research intent with fit under 0.7 |
| **ours** | We rank in the top 10 |
| **open** | No tracked competitor ranks in the top 10 |
| **fight for** | A competitor ranks, but the fit is strong (0.7 or more) and their best rank is #4 or lower |
| **not winnable** | A competitor holds it near the top and we have no clear edge |

In the full tool, "fight for" vs. "not winnable" is judged by an LLM against the company's advantages and verified notes on each competitor, with a written reason. The rule here is the fallback the tool uses until that judgment exists.

**Opportunity** ranks what to write first: √demand × fit × beatability × intent. Beatability is 1.0 when no competitor ranks, and 0.4 to 0.94 when one does, rising the lower they rank. Taking the square root of demand keeps fit in charge, because a specific phrase we can credibly own beats a broad one we can't.

In [4]:
TOP = 10
INTENT_FACTOR = {"business": 1.0, "mixed": 0.85, "research": 0.8, "consumer": 0.5, "jobs": 0.3}


def label(row):
    held = [r for r in (row.a_rank, row.b_rank) if pd.notna(r) and r <= TOP]
    if row.fit < 0.4 or row.intent in ("consumer", "jobs") or (row.intent == "research" and row.fit < 0.7):
        return "not worth it", f"fit {row.fit}, {row.intent} intent"
    if pd.notna(row.our_rank) and row.our_rank <= TOP:
        return "ours", f"we rank #{int(row.our_rank)}"
    if not held:
        return "open", "no tracked competitor in the top 10"
    best = int(min(held))
    if row.fit >= 0.7 and best >= 4:
        return "fight for", f"strong fit, best competitor only #{best}"
    return "not winnable", f"competitor holds #{best}"


def opportunity(row):
    held = [r for r in (row.a_rank, row.b_rank) if pd.notna(r) and r <= TOP]
    beat = 0.4 + 0.06 * (min(held) - 1) if held else 1.0
    if pd.notna(row.our_rank) and row.our_rank <= 3:
        beat *= 0.3  # already won: keep defending it, but it isn't the next opportunity
    return round(math.sqrt(row.demand) * row.fit * beat * INTENT_FACTOR[row.intent] * 100)


research[["label", "why"]] = research.apply(lambda r: pd.Series(label(r)), axis=1)
research["opportunity"] = research.apply(opportunity, axis=1)
plan = research.sort_values("opportunity", ascending=False)[["keyword", "label", "opportunity", "demand", "fit", "why"]]
plan.reset_index(drop=True)

,keyword,label,opportunity,demand,fit,why
0,reduce stockouts,open,88,0.78,1.00,no tracked competitor in the top 10
1,inventory carrying costs,fight for,69,0.95,1.00,"strong fit, best competitor only #9"
2,inventory forecasting software,fight for,60,0.89,1.00,"strong fit, best competitor only #5"
3,store level demand forecasting,ours,56,0.56,0.75,we rank #4
4,how to reduce stockouts in retail,open,52,0.61,0.67,no tracked competitor in the top 10
5,automated reorder point,not winnable,38,0.67,0.67,competitor holds #6
6,safety stock calculation,not worth it,25,1.00,0.67,"fit 0.67, research intent"
7,home pantry inventory app,not worth it,11,0.83,0.25,"fit 0.25, consumer intent"
8,inventory clerk jobs,not worth it,10,1.00,0.33,"fit 0.33, jobs intent"
9,inventory management for multiple warehouses,not worth it,9,0.73,0.25,"fit 0.25, business intent"


In [5]:
research.label.value_counts()

label
not worth it    5
fight for       2
open            2
ours            1
not winnable    1
Name: count, dtype: int64

**Reading it:** the top of the list is where to write first. "Reduce stockouts" leads: it fits exactly what Example Co sells, and no competitor ranks for it. "Inventory carrying costs" and "inventory forecasting software" are worth fighting for, because competitors rank but only at #9 and #5, and Example Co's fit is strong. Example Co already ranks #4 for "store level demand forecasting", so that one is ours to defend and push higher. "Safety stock calculation" has as much demand as anything on the list, but the searchers are researching and the fit is partial, so it's not worth a page. "Warehouse management system" is also in high demand, but it has nothing to do with what Example Co sells, so it scores 0.

**Why it matters for marketing:** most keyword tools rank terms by search volume. This ranks them by whether a smaller company can actually win them and whether the searcher is a buyer, which is the question that decides what the content team writes first.

## 4. Checking a draft before it's published

Each content piece in the plan gets a brief: the target keyword, the question to answer first, and the facts the writer may use. When a draft comes back, the tool checks it against what search engines and AI answers reward: the keyword in the H1 and first paragraph, a direct answer up front, question headings with short answers, an FAQ, and numbers that trace back to confirmed facts. It also flags absolute claims ("the only", "guaranteed") that a competitor or reviewer could disprove.

In [6]:
CONFIRMED_FACTS = ["Pilot retailers cut stockouts 18% in 6 months across 12 stores."]
ABSOLUTE = re.compile(r"\b(the only|first and only|everyone else|guarantee[ds]?|#1)\b", re.I)
NUMBER = re.compile(r"\$\s?\d[\d,]*(?:\.\d+)?|\d[\d,]*(?:\.\d+)?\s?%|\b\d[\d,]{2,}\b")

DRAFT = """
# How to reduce stockouts across multiple stores

To reduce stockouts across multiple stores, forecast demand for each store and item instead of the region,
and reorder from that forecast. Most stockouts come from a regional forecast that hides one store's demand.

## What causes stockouts?

A forecast that's too low for one store, late reorders, or safety stock set once and never updated.

## How much safety stock do you need?

Less than you think, if each store's forecast is accurate. Store-level forecasting cuts stockouts by 35%.

## Is Example Co the only tool that forecasts per store?

Example Co is the only forecasting tool built for multi-location retail.

## Frequently asked questions

### Can I keep my point-of-sale system?

Yes. Example Co connects to it.
"""


def check_draft(md_text, keyword):
    blocks = [b.strip() for b in re.split(r"\n\s*\n", md_text.strip()) if b.strip()]
    h1 = next((b[2:] for b in blocks if b.startswith("# ")), "")
    paragraphs = [b for b in blocks if not b.startswith("#")]
    first = paragraphs[0] if paragraphs else ""
    headings = [b.lstrip("#").strip() for b in blocks if b.startswith("#")]
    questions = [h for h in headings if h.endswith("?")]
    faq_at = next((i for i, h in enumerate(headings) if "question" in h.lower()), None)
    faq = [h for h in headings[faq_at + 1:] if h.endswith("?")] if faq_at is not None else []
    known = {n.replace(" ", "") for n in NUMBER.findall(" ".join(CONFIRMED_FACTS))}
    untraced = sorted({n for n in NUMBER.findall(md_text) if n.replace(" ", "") not in known})
    absolute = sorted({m.group(0) for m in ABSOLUTE.finditer(md_text)})
    has = lambda text: len(words(keyword) & words(text)) / len(words(keyword)) >= 0.75

    def row(name, ok, detail):
        return {"check": name, "result": "pass" if ok else "fail", "detail": detail}

    return pd.DataFrame([
        row("Keyword in the H1", has(h1), h1),
        row("Keyword in the first paragraph", has(first), first[:80] + "..."),
        row("Answer first (under 60 words)", 0 < len(first.split()) <= 60, f"{len(first.split())} words"),
        row("Question headings (3 or more)", len(questions) >= 3, f"{len(questions)} question headings"),
        row("FAQ section (3 or more questions)", len(faq) >= 3, f"{len(faq)} FAQ questions"),
        row("Numbers trace to confirmed facts", not untraced, ", ".join(untraced) or "all traced"),
        row("No absolute claims", not absolute, ", ".join(absolute) or "none"),
        row("No em dashes", "\u2014" not in md_text, f"{md_text.count(chr(0x2014))} em dashes"),
    ])


check_draft(DRAFT, "reduce stockouts across multiple stores")

,check,result,detail
0,Keyword in the H1,pass,How to reduce stockouts across multiple stores
1,Keyword in the first paragraph,pass,"To reduce stockouts across multiple stores, forecast demand for each store a..."
2,Answer first (under 60 words),pass,34 words
3,Question headings (3 or more),pass,4 question headings
4,FAQ section (3 or more questions),fail,1 FAQ questions
5,Numbers trace to confirmed facts,fail,35%
6,No absolute claims,fail,the only
7,No em dashes,pass,0 em dashes


**Reading it:** the draft gets the structure right, but three things would stop it from being published. "35%" isn't in the confirmed facts (the pilot measured 18%), "the only" is a claim a competitor could disprove, and the FAQ needs at least two more questions. A writer fixes these before the page goes live, not after a reviewer catches them.

---

**About the full tool:** about 3,700 lines of Python. It crawls each site politely (robots.txt, one request per second), scores pages on 40+ UX, SEO and AEO checks, compares benefit wordings with Google Trends, sorts competitor reviews into what buyers like and dislike, finds the third-party pages AI answers cite, and writes a content plan with a brief per piece. Judgments that need reading, not keyword matching (fit, intent, funnel stage, whether a claim is defensible), are queued for an LLM and reused on every later run, so each item is judged once.